![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/visual-nlp-workshop/blob/master/jupyter/SparkOcrDebugSparkInit.ipynb)

# Spark OCR init debugging

This notebook does not test any Spark OCR feature. Its only job is to reproduce the
`JAVA_GATEWAY_EXITED` failure seen in Jenkins when `sparkocr.start(...)` is called from
`SparkOcrPreserveOriginalFormatting.ipynb`, and to dump every piece of environment,
classpath and JVM launch information needed to diagnose it, as plain stdout so it shows
up in the Jenkins console log even when the notebook itself doesn't get saved with
outputs.

The Jenkinsfile has been temporarily changed to run only this notebook, ignoring every
other notebook under `workshop/`.

In [ ]:
import glob
import json
import os
import platform
import subprocess
import sys
import tempfile
import time
import traceback

# --nbval-lax only surfaces a cell's captured stdout when the cell itself fails;
# passing cells are silently discarded, which is why plain print() output can vanish
# from the Jenkins console even though nothing crashed. Tee everything to a plain file
# too, so it can be `cat`-ed unconditionally after the pytest run regardless of outcome.
DEBUG_LOG_PATH = os.path.abspath("spark_init_debug.log")
_debug_log_fh = open(DEBUG_LOG_PATH, "w", encoding="utf-8")
_builtin_print = print

def print(*args, **kwargs):
    _builtin_print(*args, **kwargs)
    text = kwargs.get("sep", " ").join(str(a) for a in args) + kwargs.get("end", "\n")
    _debug_log_fh.write(text)
    _debug_log_fh.flush()

print(f"Writing debug log to {DEBUG_LOG_PATH}")

def section(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)

def run(cmd, timeout=60, env=None):
    """Run cmd (list or shell string), print rc/stdout/stderr, never raise."""
    print(f"$ {cmd}")
    try:
        proc = subprocess.run(
            cmd,
            shell=isinstance(cmd, str),
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
        )
        print(f"rc={proc.returncode}")
        if proc.stdout.strip():
            print("--- stdout ---")
            print(proc.stdout.strip())
        if proc.stderr.strip():
            print("--- stderr ---")
            print(proc.stderr.strip())
        return proc.returncode, proc.stdout, proc.stderr
    except subprocess.TimeoutExpired as e:
        print(f"TIMED OUT after {timeout}s")
        if e.stdout:
            print("--- stdout (partial) ---")
            print(e.stdout)
        if e.stderr:
            print("--- stderr (partial) ---")
            print(e.stderr)
        return None, e.stdout or "", e.stderr or ""
    except Exception as e:
        print(f"FAILED TO RUN: {e}")
        return None, "", str(e)


## 1. Interpreter / OS / resources

In [ ]:
section("Python / OS")
print("sys.executable:", sys.executable)
print("sys.version:", sys.version)
print("platform:", platform.platform())
print("machine:", platform.machine())
run("nproc")
run("free -h")
run("ulimit -a")
run(["df", "-h", "."])


## 2. Environment variables

Known secret-bearing variables are only reported as *present/length*, never their value.

In [ ]:
SECRET_NAMES = {
    "SPARK_OCR_LICENSE", "JSL_OCR_LICENSE", "SPARK_NLP_LICENSE",
    "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN",
    "GITHUB_CREDS_PSW", "GITHUB_CREDS_USR", "JSL_NLP_SECRET",
    "CODEARTIFACT_AUTH_TOKEN", "NVD_API_KEY", "RELEASE_TEAMS_WEBHOOK",
    "SECRETS", "SECRET",
}

RELEVANT_PREFIXES = ("SPARK", "JAVA", "PYSPARK", "PYTHON", "JDK", "HADOOP", "SBT", "JSL", "NLP")

section("Environment variables (relevant subset)")
for key in sorted(os.environ):
    if key in SECRET_NAMES:
        val = os.environ[key]
        print(f"{key} = <redacted, present, len={len(val)}>")
    elif key.startswith(RELEVANT_PREFIXES) or key in (
        "PATH", "HOME", "USER", "TMPDIR", "CLASSPATH", "LD_LIBRARY_PATH",
        "_JAVA_OPTIONS", "JDK_JAVA_OPTIONS", "PYSPARK_SUBMIT_ARGS",
    ):
        print(f"{key} = {os.environ[key]}")


## 3. Java / Spark distribution on PATH

In [ ]:
section("java")
run("which java")
run("java -version")
run("java -XshowSettings:properties -version")

section("SPARK_HOME")
spark_home = os.environ.get("SPARK_HOME")
print("SPARK_HOME env:", spark_home)
try:
    import pyspark
    print("pyspark module dir:", os.path.dirname(pyspark.__file__))
    from pyspark.find_spark_home import _find_spark_home
    print("pyspark._find_spark_home():", _find_spark_home())
    if not spark_home:
        spark_home = _find_spark_home()
except Exception:
    traceback.print_exc()

if spark_home:
    run(["ls", "-la", os.path.join(spark_home, "bin")])
    run([os.path.join(spark_home, "bin", "spark-submit"), "--version"])


## 4. Installed python package versions

In [ ]:
section("pip freeze")
run([sys.executable, "-m", "pip", "freeze"])


In [ ]:
section("module versions")
for mod_name in ["pyspark", "py4j", "sparknlp", "sparknlp_jsl", "sparkocr", "pandas", "numpy"]:
    try:
        mod = __import__(mod_name)
        version = getattr(mod, "__version__", None)
        if version is None and hasattr(mod, "version"):
            try:
                version = mod.version()
            except Exception:
                version = "<version() call failed>"
        print(f"{mod_name}: {version}  ({mod.__file__})")
    except Exception as e:
        print(f"{mod_name}: NOT IMPORTABLE ({e})")


## 5. Jars expected by `sparkocr.start(...)`

In [ ]:
secret = ""
license = ""
jar_path = "../../target/scala-2.13/"

section("jar_path candidates under ../../target/")
for scala_dir in ["scala-2.11", "scala-2.12", "scala-2.13"]:
    p = os.path.abspath(os.path.join("..", "..", "target", scala_dir))
    exists = os.path.isdir(p)
    print(p, "-> exists:", exists, " is_symlink:", os.path.islink(p))
    if exists:
        for f in sorted(glob.glob(os.path.join(p, "*.jar"))):
            size = os.path.getsize(f)
            with open(f, "rb") as fh:
                magic = fh.read(4)
            print(f"  {f}  size={size}  magic_ok={magic[:2] == b'PK'}")


## 6. Minimal bare SparkSession (no Spark OCR / Spark NLP jars or packages)

Isolates whether the Java gateway can start **at all** in this environment,
independent of Spark OCR / Spark NLP jars and package resolution.

In [ ]:
section("Bare SparkSession.getOrCreate()")
try:
    from pyspark.sql import SparkSession
    bare_spark = SparkSession.builder.appName("debug-bare").master("local[2]").getOrCreate()
    print("OK - bare SparkSession started")
    print(bare_spark.sparkContext.getConf().getAll())
    bare_spark.stop()
except Exception:
    print("FAILED to start bare SparkSession")
    traceback.print_exc()


## 7. SparkSession with only the local Spark OCR jar (no `spark.jars.packages`)

Isolates whether the assembly jar itself (built for this branch/scala version) is the
problem, independent of Ivy/Maven package resolution for spark-nlp.

In [ ]:
section("SparkSession with local OCR jar only")
try:
    from pyspark.sql import SparkSession
    ocr_jar_candidates = glob.glob(os.path.abspath(os.path.join(jar_path, "spark-ocr-assembly-*.jar")))
    print("candidate jars:", ocr_jar_candidates)
    if ocr_jar_candidates:
        jars_spark = (
            SparkSession.builder
            .appName("debug-local-jar")
            .master("local[2]")
            .config("spark.jars", ocr_jar_candidates[0])
            .getOrCreate()
        )
        print("OK - SparkSession with local OCR jar started")
        jars_spark.stop()
    else:
        print("No local OCR assembly jar found under jar_path, skipping this check")
except Exception:
    print("FAILED to start SparkSession with local OCR jar")
    traceback.print_exc()


## 8. Manual `spark-submit` with the exact `--packages` coordinate `start()` uses

`sparkocr.start()` adds `com.johnsnowlabs.nlp:spark-nlp_2.13:<nlp_version>` to
`spark.jars.packages` with no custom `spark.jars.repositories`, so Ivy falls back to
Spark's default repository list (Maven Central, Spark Packages, etc). If the CI
container has restricted network egress, or one of the default repos is slow/unreachable,
Ivy resolution can hang past pyspark's fixed gateway-launch timeout, which surfaces as
the exact same `JAVA_GATEWAY_EXITED` / "process exited before sending port number" error
even though the JVM never actually crashed - it just didn't finish in time. Running the
same `--packages` resolution directly, with a generous timeout and full stderr captured,
tells us whether that's what's happening here.

In [ ]:
section("Manual spark-submit with --packages com.johnsnowlabs.nlp:spark-nlp_2.13:...")
if spark_home:
    submit = os.path.join(spark_home, "bin", "spark-submit")
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write("print('SPARK_DEBUG_PROBE_OK')\n")
        probe_script = f.name

    packages = "com.johnsnowlabs.nlp:spark-nlp_2.13:6.4.2"
    start_ts = time.time()
    run(
        [submit, "--master", "local[2]", "--conf", "spark.ui.enabled=false",
         "--packages", packages, probe_script],
        timeout=300,
    )
    print(f"elapsed: {time.time() - start_ts:.1f}s")
else:
    print("Could not resolve SPARK_HOME, skipping")


## 9. The actual failing call: `sparkocr.start(secret=secret, jar_path=jar_path)`

Same call as in `SparkOcrPreserveOriginalFormatting.ipynb`, wrapped so the full
traceback (and not just the notebook's truncated error) reaches stdout.

In [ ]:
section("sparkocr.start(...)")
if license:
    os.environ['SPARK_OCR_LICENSE'] = license

os.environ["SPARK_PRINT_LAUNCH_COMMAND"] = "1"  # pyspark prints the exact spark-submit command it runs

try:
    from sparkocr import start
    spark = start(secret=secret, jar_path=jar_path)
    print("OK - sparkocr.start() succeeded")
    spark.stop()
except Exception:
    print("FAILED - sparkocr.start() raised")
    traceback.print_exc()


## 10. JVM crash logs, if any

In [ ]:
section("hs_err_pid*.log search")
found_any = False
for base in ["/tmp", os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), ".."))]:
    for f in glob.glob(os.path.join(base, "hs_err_pid*.log")):
        found_any = True
        print(f"--- {f} ---")
        with open(f, encoding="utf-8", errors="replace") as fh:
            print(fh.read())
if not found_any:
    print("No hs_err_pid*.log files found")


In [ ]:
section("done")
print(f"full debug log written to {DEBUG_LOG_PATH}")
_debug_log_fh.flush()
_debug_log_fh.close()
